# 04 · Multi-source pipeline: five sources → one customer
```
Turso                 Landing (Parquet)                 Bronze / Silver (per source)            Silver (combined)            Gold
crm_accounts      →   landing/crm/crm_accounts      →   bronze.crm_accounts      → silver.crm_accounts      ─┐
billing_customers →   landing/billing/...           →   bronze.billing_customers → silver.billing_customers ─┤
support_contacts  →   landing/support/...           →   ...                                                 ├→ silver.customer_unified → gold.dim_customer_360
web_visitors      →   landing/web/...               →   ...                                                 │   (match on email,
marketing_leads   →   landing/marketing/...         →   ...                                                ─┘    precedence per attribute)
```
Landing, Bronze and Silver keep each source separate, so one bad feed never blocks the others. The combine step matches people **by normalised email** and picks each attribute from the highest-priority source that has it (see `ATTR`). Reuses every helper from `00_config` (stage, clean, merge, watermarks); this notebook only adds metadata and the combine logic.

In [1]:
%run -i ./00_config.ipynb

## Register the five sources
Same metadata format as before. `digits` is a small DuckDB macro (last 10 digits of a phone number) used as a column transform. Rows without an email cannot be matched, so `email` is *required* and such rows go to `silver._quarantine`.

In [2]:
ENTITIES.update({
  "crm_accounts": dict(keys=["account_id"], order_by="updated_at",
    columns=[("account_id","BIGINT"),("full_name","VARCHAR"),("email","VARCHAR","lower"),("phone","VARCHAR","digits"),
             ("segment","VARCHAR","lower"),("country","VARCHAR","upper"),("owner","VARCHAR","lower"),("updated_at","TIMESTAMP")],
    required=["account_id","email","updated_at"], rules=[]),
  "billing_customers": dict(keys=["billing_id"], order_by="updated_at",
    columns=[("billing_id","BIGINT"),("email","VARCHAR","lower"),("billing_name","VARCHAR"),("plan","VARCHAR","lower"),
             ("status","VARCHAR","lower"),("mrr","DECIMAL(10,2)"),("country","VARCHAR","upper"),("updated_at","TIMESTAMP")],
    required=["billing_id","email","updated_at"], rules=["mrr >= 0"]),
  "support_contacts": dict(keys=["contact_id"], order_by="updated_at",
    columns=[("contact_id","BIGINT"),("email","VARCHAR","lower"),("name","VARCHAR"),("phone","VARCHAR","digits"),
             ("tier","VARCHAR","lower"),("updated_at","TIMESTAMP")],
    required=["contact_id","email","updated_at"], rules=[]),
  "web_visitors": dict(keys=["visitor_id"], order_by="updated_at",
    columns=[("visitor_id","BIGINT"),("email","VARCHAR","lower"),("first_seen","TIMESTAMP"),("last_seen","TIMESTAMP"),
             ("visits","INTEGER"),("updated_at","TIMESTAMP")],
    required=["visitor_id","email","updated_at"], rules=["visits >= 0"]),
  "marketing_leads": dict(keys=["lead_id"], order_by="updated_at",
    columns=[("lead_id","BIGINT"),("email","VARCHAR","lower"),("name","VARCHAR"),("lead_source","VARCHAR","lower"),
             ("lead_status","VARCHAR","lower"),("marketing_consent","VARCHAR","upper"),("updated_at","TIMESTAMP")],
    required=["lead_id","email","updated_at"], rules=[]),
})
SOURCES = ["crm_accounts", "billing_customers", "support_contacts", "web_visitors", "marketing_leads"]
SOURCE_OF = {"crm_accounts": "crm", "billing_customers": "billing", "support_contacts": "support",
             "web_visitors": "web", "marketing_leads": "marketing"}

# landing tree: landing/<source system>/<entity>/batch_id=N/part-*.parquet
_LANDING_BASE = LAKE / "landing"
def stage_source(entity):
    global LANDING
    LANDING = _LANDING_BASE / SOURCE_OF[entity]
    try: stage(entity)
    finally: LANDING = _LANDING_BASE

In [3]:
open_warehouse()
con.execute("CREATE OR REPLACE MACRO digits(x) AS right(regexp_replace(x, '[^0-9]', '', 'g'), 10)")
src = source_conn()
new_batch()
print(f"run batch {BATCH_ID} | source: {'Turso' if TURSO_URL else 'local SQLite stand-in'}")

run batch 2 | source: Turso


## 1 · Stage and 2 · Clean, one source at a time

In [4]:
for e in SOURCES: stage_source(e)

  crm_accounts extracted    60 rows (since the beginning),    60 new in bronze
  billing_customers extracted    47 rows (since the beginning),    47 new in bronze
  support_contacts extracted    34 rows (since the beginning),    34 new in bronze
  web_visitors extracted    31 rows (since the beginning),    31 new in bronze
  marketing_leads extracted    35 rows (since the beginning),    35 new in bronze


In [5]:
for e in SOURCES: clean_to_silver(e)

  crm_accounts    56 valid latest rows ->    56 inserted/updated,   4 quarantined
  billing_customers    46 valid latest rows ->    46 inserted/updated,   1 quarantined
  support_contacts    34 valid latest rows ->    34 inserted/updated,   0 quarantined
  web_visitors    27 valid latest rows ->    27 inserted/updated,   4 quarantined
  marketing_leads    35 valid latest rows ->    35 inserted/updated,   0 quarantined


## 3 · Combine: one row per person
`affected` = emails whose row changed in any source since the last run (every email on the first run or in `full` mode). For those emails, take the latest Silver row from each source, join them on email, and build each attribute with `coalesce` in the order listed in `ATTR` (first source that has a value wins). The result is merged into `silver.customer_unified`; only people whose values changed are rewritten.

> Limitation: an email *change* in a source is an identity change. This example treats the new email as a new person; handling it needs an identity-resolution step (a stable person id).

In [6]:
SRC_TABLE = {"crm": ("crm_accounts", "account_id"), "billing": ("billing_customers", "billing_id"),
             "support": ("support_contacts", "contact_id"), "web": ("web_visitors", "visitor_id"),
             "mkt": ("marketing_leads", "lead_id")}

ATTR = {   # unified column -> source columns in precedence order (first non-null wins)
  "full_name": ["crm.full_name", "billing.billing_name", "support.name", "mkt.name"],
  "phone": ["support.phone", "crm.phone"],
  "segment": ["crm.segment"],
  "country": ["crm.country", "billing.country"],
  "plan": ["billing.plan"], "billing_status": ["billing.status"], "mrr": ["billing.mrr"],
  "support_tier": ["support.tier"],
  "first_seen": ["web.first_seen"], "last_seen": ["web.last_seen"], "visits": ["web.visits"],
  "lead_source": ["mkt.lead_source"], "lead_status": ["mkt.lead_status"], "marketing_consent": ["mkt.marketing_consent"],
  "crm_account_id": ["crm.account_id"], "billing_id": ["billing.billing_id"], "support_contact_id": ["support.contact_id"],
  "web_visitor_id": ["web.visitor_id"], "marketing_lead_id": ["mkt.lead_id"],
}
FLAGS = {"in_crm": "crm", "in_billing": "billing", "in_support": "support", "in_web": "web", "in_marketing": "mkt"}

UCOLS = [("customer_key","VARCHAR"),("email","VARCHAR"),("full_name","VARCHAR"),("phone","VARCHAR"),("segment","VARCHAR"),
  ("country","VARCHAR"),("plan","VARCHAR"),("billing_status","VARCHAR"),("mrr","DECIMAL(10,2)"),("support_tier","VARCHAR"),
  ("first_seen","TIMESTAMP"),("last_seen","TIMESTAMP"),("visits","INTEGER"),("lead_source","VARCHAR"),("lead_status","VARCHAR"),
  ("marketing_consent","VARCHAR"),("crm_account_id","BIGINT"),("billing_id","BIGINT"),("support_contact_id","BIGINT"),
  ("web_visitor_id","BIGINT"),("marketing_lead_id","BIGINT"),("in_crm","BOOLEAN"),("in_billing","BOOLEAN"),
  ("in_support","BOOLEAN"),("in_web","BOOLEAN"),("in_marketing","BOOLEAN"),("source_count","INTEGER")]

def unify_customers():
    wm = get_wm("silver.customer_unified")
    if LOAD_MODE == "full" or wm is None:
        affected = " UNION ".join(f"SELECT email FROM silver.{t}" for t, _ in SRC_TABLE.values())
    else:
        affected = " UNION ".join(f"SELECT email FROM silver.{t} WHERE _silver_batch > {int(wm)}" for t, _ in SRC_TABLE.values())
    latest = ",\n".join(f"""{a} AS (SELECT * FROM silver.{t} WHERE email IN (SELECT email FROM affected)
                 QUALIFY row_number() OVER (PARTITION BY email ORDER BY updated_at DESC, {k} DESC) = 1)"""
                        for a, (t, k) in SRC_TABLE.items())
    sel = {"customer_key": "md5(e.email)", "email": "e.email"}
    for col, srcs in ATTR.items(): sel[col] = "coalesce(" + ", ".join(srcs) + ")"
    for flag, a in FLAGS.items():   sel[flag] = f"{a}.email IS NOT NULL"
    sel["source_count"] = " + ".join(f"CAST({a}.email IS NOT NULL AS INTEGER)" for a in FLAGS.values())
    joins = " ".join(f"LEFT JOIN {a} ON {a}.email = e.email" for a in SRC_TABLE)
    con.execute(f"""CREATE OR REPLACE TEMP TABLE _unified AS
        WITH affected AS ({affected}), {latest}
        SELECT {', '.join(f'{sel[c]} AS {c}' for c, _ in UCOLS)} FROM affected e {joins}""")
    con.execute(f"""CREATE TABLE IF NOT EXISTS silver.customer_unified
        ({', '.join(f'{n} {t}' for n, t in UCOLS)}, _silver_batch BIGINT, PRIMARY KEY (customer_key))""")
    n = con.execute("SELECT count(*) FROM _unified").fetchone()[0]
    changed = merge("silver.customer_unified", "_unified", ["customer_key"], [c for c, _ in UCOLS], "_silver_batch")
    set_wm("silver.customer_unified", BATCH_ID)
    print(f"  customer_unified  {n:5d} people re-evaluated -> {changed:5d} inserted/updated")

unify_customers()

  customer_unified     96 people re-evaluated ->    96 inserted/updated


## 4 · Merge into Gold: `gold.dim_customer_360`
Same incremental pattern as before: only people changed in `silver.customer_unified` since the last Gold load are merged (`_silver_batch` watermark). `lifecycle` is derived here. `gold.dim_customer_360.crm_account_id` is the bridge back to your existing sales data.

In [7]:
DIM360 = ["email","full_name","phone","segment","country","lifecycle","plan","billing_status","mrr","support_tier",
          "first_seen","last_seen","visits","lead_source","lead_status","marketing_consent","crm_account_id",
          "in_crm","in_billing","in_support","in_web","in_marketing","source_count"]

LIFECYCLE = """CASE WHEN in_billing AND billing_status IN ('active', 'past_due') THEN 'customer'
                    WHEN in_billing THEN 'former customer'
                    WHEN in_crm THEN 'prospect'
                    WHEN in_marketing THEN 'lead'
                    ELSE 'visitor or contact' END"""

def load_dim_customer360():
    tgt, seq = "gold.dim_customer_360", "gold.seq_customer360_sk"
    types = dict(UCOLS); types["lifecycle"] = "VARCHAR"
    con.execute(f"CREATE SEQUENCE IF NOT EXISTS {seq} START 1")
    con.execute(f"""CREATE TABLE IF NOT EXISTS {tgt} (customer_sk BIGINT PRIMARY KEY, customer_key VARCHAR UNIQUE NOT NULL,
                    {', '.join(f'{c} {types[c]}' for c in DIM360)}, _gold_batch BIGINT)""")
    con.execute(f"INSERT INTO {tgt} (customer_sk, customer_key) SELECT -1, '-1' WHERE NOT EXISTS (SELECT 1 FROM {tgt} WHERE customer_sk = -1)")
    pred = delta_predicate("dim_customer_360", "customer_unified")
    cols = ["customer_key", *DIM360]
    sel = ", ".join(f"{LIFECYCLE} AS lifecycle" if c == "lifecycle" else c for c in cols)
    changed = merge(tgt, f"(SELECT {sel} FROM silver.customer_unified s WHERE {pred})", ["customer_key"], cols,
                    "_gold_batch", sk=("customer_sk", f"nextval('{seq}')"))
    set_wm("gold.dim_customer_360", PENDING["dim_customer_360"])
    con.execute("""CREATE OR REPLACE VIEW gold.v_customer_source_overlap AS
        SELECT concat_ws('+', CASE WHEN in_crm THEN 'crm' END, CASE WHEN in_billing THEN 'billing' END,
                              CASE WHEN in_support THEN 'support' END, CASE WHEN in_web THEN 'web' END,
                              CASE WHEN in_marketing THEN 'marketing' END) AS sources, count(*) AS customers
        FROM gold.dim_customer_360 WHERE customer_sk <> -1 GROUP BY 1 ORDER BY 2 DESC""")
    print(f"  dim_customer_360  {changed:5d} inserted/updated")

load_dim_customer360()

  dim_customer_360     96 inserted/updated


## 5 · Checks

In [8]:
display(con.execute("SELECT * FROM ctl.watermark WHERE name LIKE '%customer%' OR name LIKE '%crm%' OR name LIKE '%billing%' ORDER BY name").df())
display(con.execute("SELECT entity, count(*) AS quarantined FROM silver._quarantine WHERE entity IN (" + ",".join(f"'{s}'" for s in SOURCES) + ") GROUP BY 1 ORDER BY 1").df())
display(con.execute("SELECT lifecycle, count(*) AS customers FROM gold.dim_customer_360 WHERE customer_sk <> -1 GROUP BY 1 ORDER BY 2 DESC").df())
display(con.execute("SELECT * FROM gold.v_customer_source_overlap").df())

,name,value,updated_ts
0,extract.billing_customers,2026-10-31 20:29:53,2026-11-18 19:05:35.211228
1,extract.crm_accounts,2026-10-30 14:13:55,2026-11-18 19:05:35.181948
2,extract.customers,2026-10-30 14:13:55,2026-11-18 17:24:35.489720
3,gold.dim_customer,1,2026-11-18 17:24:37.839766
4,gold.dim_customer_360,2,2026-11-18 19:05:37.704804
5,silver.billing_customers,2,2026-11-18 19:05:37.181755
6,silver.crm_accounts,2,2026-11-18 19:05:37.143307
7,silver.customer_unified,2,2026-11-18 19:05:37.549549
8,silver.customers,1,2026-11-18 17:24:36.795456


,entity,quarantined
0,billing_customers,1
1,crm_accounts,4
2,web_visitors,4


,lifecycle,customers
0,customer,38
1,lead,25
2,prospect,18
3,visitor or contact,10
4,former customer,5


,sources,customers
0,marketing,25
1,crm+billing+support,12
2,crm+billing,10
3,web,6
4,crm+billing+support+web,6
5,billing,5
6,crm,5
7,crm+billing+web,4
8,support,4
9,crm+web,3


In [9]:
# Where sources disagree, the precedence rules decide: CRM name vs billing name for the same email
display(con.execute("""
  SELECT c.email, c.full_name AS crm_name, b.billing_name, d.full_name AS gold_name
  FROM silver.crm_accounts c JOIN silver.billing_customers b USING (email)
  JOIN gold.dim_customer_360 d ON d.email = c.email
  WHERE lower(c.full_name) <> lower(b.billing_name) LIMIT 8""").df())
display(con.execute("SELECT * FROM gold.dim_customer_360 WHERE source_count = 5 LIMIT 3").df())

,email,crm_name,billing_name,gold_name
0,wen.novak55@example.com,Wen Novak,W. Novak,Wen Novak
1,frank.meyer54@example.com,Frank Meyer,F. Meyer,Frank Meyer
2,pavel.cohen25@example.com,Pavel Cohen,P. Cohen,Pavel Cohen
3,vik.jones50@example.com,Vik Jones,V. Jones,Vik Jones


,customer_sk,customer_key,email,full_name,phone,segment,country,lifecycle,plan,billing_status,...,lead_status,marketing_consent,crm_account_id,in_crm,in_billing,in_support,in_web,in_marketing,source_count,_gold_batch
0,1,c39a9478a1aac2a49af921aacdeb5677,uma.garcia20@example.com,Uma Garcia,6172919159,retail,GB,customer,pro,active,...,new,N,20,True,True,True,True,True,5,2


Release the DuckDB lock so you can inspect the file from other notebooks:

In [10]:
con.close()

In [1]:
import duckdb
c = duckdb.connect("lake/warehouse.duckdb", read_only=True)
c.sql("SELECT * FROM gold.dim_customer_360 LIMIT 10").show()
c.close()

┌─────────────┬──────────────────────────────────┬───────────────────────────┬──────────────┬────────────┬────────────┬─────────┬─────────────────┬────────────┬────────────────┬───────────────┬──────────────┬─────────────────────┬─────────────────────┬────────┬─────────────┬─────────────┬───────────────────┬────────────────┬─────────┬────────────┬────────────┬─────────┬──────────────┬──────────────┬─────────────┐
│ customer_sk │           customer_key           │           email           │  full_name   │   phone    │  segment   │ country │    lifecycle    │    plan    │ billing_status │      mrr      │ support_tier │     first_seen      │      last_seen      │ visits │ lead_source │ lead_status │ marketing_consent │ crm_account_id │ in_crm  │ in_billing │ in_support │ in_web  │ in_marketing │ source_count │ _gold_batch │
│    int64    │             varchar              │          varchar          │   varchar    │  varchar   │  varchar   │ varchar │     varchar     │  varchar   │    va

In [2]:
import duckdb
c = duckdb.connect("lake/warehouse.duckdb", read_only=True)
c.sql("SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema IN ('bronze','silver','gold','ctl')ORDER BY 1, 2").show()
c.close()

┌──────────────┬───────────────────┐
│ table_schema │    table_name     │
│   varchar    │      varchar      │
├──────────────┼───────────────────┤
│ bronze       │ billing_customers │
│ bronze       │ crm_accounts      │
│ bronze       │ customers         │
│ bronze       │ marketing_leads   │
│ bronze       │ products          │
│ bronze       │ sales             │
│ bronze       │ support_contacts  │
│ bronze       │ web_visitors      │
│ ctl          │ batch_log         │
│ ctl          │ watermark         │
│  ·           │     ·             │
│  ·           │     ·             │
│  ·           │     ·             │
│ silver       │ _quarantine       │
│ silver       │ billing_customers │
│ silver       │ crm_accounts      │
│ silver       │ customer_unified  │
│ silver       │ customers         │
│ silver       │ marketing_leads   │
│ silver       │ products          │
│ silver       │ sales             │
│ silver       │ support_contacts  │
│ silver       │ web_visitors      │
└